# Acquisition Optimization Workflow

acquisition functionを作った後、次に必要なのは**どのsearch semanticsとnumerical backendで最大化するか**です。

```text
surrogate → acquisition
                ↓
         search strategy
                ↓
       optimizer backend
                ↓
             candidate
```

このNotebookではcanonical continuous pathを中心に、joint q-batch、fixed features、candidate constraints、named backend capabilityを実行確認します。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import SingleTaskGP
from robotorchan.optim import (
    CandidateConstraints,
    get_optimizer_capabilities,
    optimize_acqf,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Fit a surrogate and construct the acquisition

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    x0 = X[..., :1]
    x1 = X[..., 1:2]
    return torch.sin(2 * torch.pi * x0) + 0.35 * torch.cos(2 * torch.pi * x1)


train_X = torch.rand(14, 2)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

## 3. Canonical continuous optimization

default `optimizer="botorch"`はBoTorch/SciPy multi-start optimizationへ薄く委譲します。BoTorchのacquisition vocabularyを置き換えません。

In [ ]:
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([1, 2])
print("candidate:", candidate)
print("acquisition value:", value)

## 4. Joint q-batch optimization

`q=2`では`[2, d]`全体が1つのdecision variableです。「q=1 optimizerを2回呼ぶ」こととは同じではありません。

In [ ]:
batch_candidate, batch_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
)

assert batch_candidate.shape == torch.Size([2, 2])
print("joint q=2 candidate:\n", batch_candidate)
print("joint acquisition value:", batch_value)

## 5. Fixed features

既知coordinateを固定することはmodelやobjectiveの責務ではなくcandidate optimizationのcontractです。

In [ ]:
fixed_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    fixed_features={1: 0.4},
)

torch.testing.assert_close(fixed_candidate[:, 1], torch.tensor([0.4]))
print("fixed-feature candidate:", fixed_candidate)

## 6. Candidate-space linear constraints

`CandidateConstraints`は既知の入力制約です。BoTorch conventionに合わせ、不等式は`sum(coefficients * X[indices]) >= rhs`です。ここでは`x0 + x1 >= 1.1`を課します。

In [ ]:
constraint = (
    torch.tensor([0, 1], dtype=torch.long),
    torch.tensor([1.0, 1.0]),
    1.1,
)
constraints = CandidateConstraints(
    inequality_constraints=(constraint,),
)

constrained_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    constraints=constraints,
)

assert constrained_candidate.sum() >= 1.1 - 1e-6
print("constrained candidate:", constrained_candidate)

## 7. Inter-point constraint requires joint q semantics

inter-point constraintはcandidate row単独では評価できません。ここではq=2の2候補について`x0(candidate 0) + x0(candidate 1) >= 1.2`を課します。

In [ ]:
joint_indices = torch.tensor([[0, 0], [1, 0]], dtype=torch.long)
joint_coefficients = torch.tensor([1.0, 1.0])
joint_constraints = CandidateConstraints(
    inequality_constraints=((joint_indices, joint_coefficients, 1.2),),
)

joint_candidate, _ = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=4,
    raw_samples=64,
    constraints=joint_constraints,
)

assert joint_candidate[:, 0].sum() >= 1.2 - 1e-6
print("joint constrained candidates:\n", joint_candidate)

## 8. Backend capability is part of the runtime contract

backend名だけで互換と判断せず、variable type、constraints、q、sequential、fixed featuresなどのcapabilityを確認します。

In [ ]:
for name in ["botorch", "sobol", "de", "ga", "pso"]:
    capabilities = get_optimizer_capabilities(name)
    print(
        name,
        {
            "continuous": capabilities.continuous,
            "q_batch": capabilities.q_batch,
            "fixed_features": capabilities.fixed_features,
            "linear_constraints": capabilities.linear_inequality_constraints,
        },
    )

## 9. A gradient-free backend keeps the acquisition contract

backendを変えても入力は同じBoTorch `AcquisitionFunction`です。ここでは軽量なSobol searchを比較します。

In [ ]:
sobol_candidate, sobol_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    optimizer="sobol",
    raw_samples=256,
    seed=123,
)

assert sobol_candidate.shape == torch.Size([1, 2])
print("Sobol-search candidate:", sobol_candidate)
print("Sobol-search acquisition value:", sobol_value)

## 10. Candidate evaluation closes the optimization layer

In [ ]:
candidate_Y = response(candidate)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("candidate Y:", candidate_Y)
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 11. Selection map

| Situation | Optimization path |
| --- | --- |
| smooth continuous acquisition | BoTorch multi-start |
| joint q-batch | q-aware backend; preserve `[q, d]` semantics |
| known candidate constraints | constraint-capable backend |
| integer/categorical/mixed | structured-space capable backend |
| unavailable candidate gradients | derivative-free / sampling backend |
| one-shot qKG/qMFKG | one-shot-aware optimization path |
| high-dimensional embedding | search strategy then backend |
| TuRBO | trust-region strategy then backend / Thompson path |
| vector numerical objective | specialist optimizer such as NSGA-II when appropriate |

## 12. Important boundaries

- acquisition optimizationはsurrogate modelを再定義しません。
- search strategyとnumerical optimizer backendは別レイヤーです。
- candidate/input constraintsとprobabilistic output constraintsを混同しません。
- joint q-batch supportとsequential optimization supportは別capabilityです。
- mixed variableをcontinuous relaxationとして黙って扱いません。
- one-shot acquisitionはauxiliary variablesを含むため通常のq-batchだけでは互換性を判断できません。
- TuRBOはoptimizer backend名ではなくstateful search geometryです。
- unsupported combinationはsilent approximationではなくfail-fastさせます。

## 13. Related documentation

- `docs/optimization/optimizer-architecture.md`
- `docs/optimization/acquisition-integration.md`
- `docs/theory/optimization/12_selection_guide.md`
- `docs/theory/optimization/05_constraints.md`
- `docs/theory/optimization/08_batch_one_shot.md`

Phase 18以降ではBatch / Async / Fantasization、Initialization、TuRBOなど、ここで分離したspecialized semanticsを個別に扱います。